# SingleTaskGP: 最小のEnd-to-End Bayesian Optimization

`robotorchan.models.SingleTaskGP` を使い、1次元のブラックボックス関数を最大化します。
モデルを作って予測するだけでなく、候補生成・評価・データ更新までを実行します。

## このNotebookで確認すること

- `SingleTaskGP` の構築、`fit()`による学習
- posterior mean / uncertainty の確認
- `qLogExpectedImprovement` の構築
- acquisition optimization による候補生成
- 候補評価と短いBO loop

関連理論:

- [Bayesian Optimization](../../docs/theory/01_bayesian_optimization.md)
- [Gaussian Process](../../docs/theory/02_gaussian_process.md)
- [Improvement-based acquisition](../../docs/theory/acquisition/02_improvement.md)

モデル選択は [Model selection](../../docs/getting_started/model_selection.md) も参照してください。

```text
Data -> Model -> Fit -> Posterior -> Acquisition -> Optimization -> Candidate -> Evaluation
                                                                      |
                                                                      +-> Data update -> repeat
```

この例は最大化問題です。Samplingや明示的なObjective / PosteriorTransformは、
この単一出力の`qLogExpectedImprovement`例では追加しません。必要なレイヤーだけを示します。

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.logei import qLogExpectedImprovement

from robotorchan.models import SingleTaskGP
from robotorchan.optim import optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### なぜベイズ最適化を使うのか

目的関数を1回評価するのに時間や費用がかかる場合、すべての入力を試すのは現実的ではありません。ベイズ最適化（BO）は、**少数の観測データから次に評価する入力を決める**方法です。

このNotebookでは、実際の実験装置の代わりに既知の数式 `objective(x)` を使います。ただしBO側にはこの数式を渡しません。BOが知るのは、評価済みの `train_X` と `train_Y` だけです。

基本的な流れは、(1) 観測、(2) GPを学習、(3) 獲得関数で次の点を選択、(4) 評価、(5) データを追加、という反復です。

## 2. Problem setup

探索区間 $[0, 1]$ で、次の滑らかなブラックボックス関数を**最大化**します。

In [ ]:
def objective(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * x) + 0.2 * torch.cos(6 * torch.pi * x)


bounds = torch.tensor([[0.0], [1.0]])

## 3. Training data

最初は10点の観測から開始します。`train_X`は`n x d`、`train_Y`は`n x 1`です。

In [ ]:
train_X = torch.linspace(0.05, 0.95, 10).unsqueeze(-1)
train_Y = objective(train_X)

print("train_X:", train_X.shape)
print("train_Y:", train_Y.shape)
print("best observed value:", train_Y.max().item())

### GPが学習するもの

ガウス過程（Gaussian Process; GP）は、未知の関数 $f(x)$ に対して、観測済みの点を条件とした予測分布を与えます。代表的には

$$f(x)\mid\mathcal D\sim\mathcal N\bigl(\mu(x),\sigma^2(x)\bigr)$$

と書きます。$\mu(x)$ は予測平均、$\sigma^2(x)$ は予測分散、$\mathcal D$ はこれまでの観測データです。**平均が高い場所**だけでなく、**まだ不確かな場所**も区別できることが、次の実験を選ぶ際に重要です。

`train_X` は入力（10行×1変数）、`train_Y` は観測値（10行×1出力）です。`SingleTaskGP` はこの組から入力と出力の関係を学習します。

## 4. Model construction

robotorchan wrapperはBoTorchの予測契約を保ちながら、raw inputと`make_mll()`を提供します。

In [ ]:
model = SingleTaskGP(train_X=train_X, train_Y=train_Y)

print(type(model).__name__)
print("supports_mll:", model.supports_mll)
print("raw_data_names:", model.raw_data_names)
print("raw_train_X shape:", model.raw_train_X.shape)
print("raw_train_Y shape:", model.raw_train_Y.shape)

### `fit()` と `make_mll()` の役割

`fit()` はモデルのハイパーパラメータ（長さ尺度・出力スケール・ノイズなど）を、観測データに合うように推定します。Exact GPでは一般に周辺対数尤度（marginal log likelihood; MLL）を最大化します。

`make_mll()` は学習に用いるMLLオブジェクトを構築するためのAPIです。このNotebookでは学習そのものは `model.fit()` に任せ、`make_mll()` は構成を確認するためだけに呼びます。

## 5. Model fitting

このモデルは`fit()`に対応しています。学習後に`make_mll()`でMLLの型を確認します。

In [ ]:
model.fit()
mll = model.make_mll()
model.eval()

print(type(mll).__name__)

### 95%区間の読み方

事後平均は1本の予測曲線ですが、GPは各入力で予測の不確実性も返します。正規分布の近似的な95%区間は $\mu(x)\pm1.96\sigma(x)$ で、下の `confidence_region()` は約2標準偏差の区間を返します。

この区間は**GPモデルとその仮定の下での潜在関数値の不確実性**を示します。頻度論的な「真の関数が95%の確率で入る保証」ではなく、観測ノイズを含む将来の測定値の予測区間とも区別してください。

今回の目的関数は決定論的（同じ入力なら同じ出力）ですが、GPの観測モデルには数値安定性などのためのノイズ成分が存在し得ます。

## 6. Posterior inspection

密なgrid上でposterior meanと95%区間を確認します。

In [ ]:
test_X = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)

with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean.squeeze(-1)
    lower, upper = posterior.mvn.confidence_region()

lower = lower.squeeze(-1)
upper = upper.squeeze(-1)

print("posterior mean:", posterior.mean.shape)
print("posterior variance:", posterior.variance.shape)

## 7. Posterior visualization

In [ ]:
plt.figure(figsize=(8, 4))
plt.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="observed")
plt.plot(test_X.squeeze(-1), mean, label="posterior mean")
plt.fill_between(
    test_X.squeeze(-1),
    lower,
    upper,
    alpha=0.2,
    label="95% confidence interval",
)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

### なぜExpected Improvement（EI）を使うのか

EIは、現在の最良観測値 $f_{\mathrm{best}}$ をどれだけ上回りそうかという**改善量の期待値**です。

$$\operatorname{EI}(x)=\mathbb E\left[\max\bigl(f(x)-f_{\mathrm{best}},0\bigr)\mid\mathcal D\right]$$

予測平均が高い点だけでなく、不確実性が大きく改善の可能性がある点にも価値を与えるため、**活用（exploitation）と探索（exploration）**のバランスを取れます。

`qLogExpectedImprovement` はEIの対数を用いて数値的に安定化したBoTorchの獲得関数です。ここでは `q=1` なので次の候補は1点です。

## 8. Acquisition function

現在の最大観測値を`best_f`として`qLogExpectedImprovement`を構築します。ここでは単一出力をそのまま最大化するため、明示的なObjective / PosteriorTransformは不要です。

In [ ]:
best_f = train_Y.max()
acqf = qLogExpectedImprovement(model=model, best_f=best_f)

print("best_f:", best_f.item())

### 獲得関数の最適化と目的関数の評価は別物

`optimize_acqf()` は、学習済みGPを使って**獲得関数が大きい入力**を探します。この段階では真の目的関数を評価しません。戻り値 `candidate` は提案入力、`acq_value` は獲得関数値です。

その後の `objective(candidate)` が実験・シミュレーションに相当する**高コストな評価**です。`acq_value` と `candidate_Y` は意味も単位も異なり、比較してはいけません。

## 9. Acquisition optimization and candidate

`robotorchan.optim.optimize_acqf`の既定backendはBoTorchです。標準的な連続探索ではBoTorchのcandidate optimizationをそのまま利用できます。

In [ ]:
candidate, acq_value = optimize_acqf(
    acq_function=acqf,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
candidate_Y = objective(candidate)

print("candidate:", candidate)
print("acquisition value:", acq_value)
print("objective(candidate):", candidate_Y)

## 10. Candidate evaluation and data update

実環境ではここが実験・シミュレーションに相当します。評価値を得たら学習データへ追加します。

In [ ]:
updated_X = torch.cat([train_X, candidate], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y], dim=0)

print("before:", train_X.shape, train_Y.shape)
print("after:", updated_X.shape, updated_Y.shape)

### BOループで更新されるもの

各反復で新しい `loop_X` / `loop_Y` からGPを学習し直し、最新の最良観測値を使って獲得関数を作り直します。したがって候補は毎回、**追加された観測結果を反映して**決まります。

実際の実験では `objective(next_X)` を装置・解析ソフトウェアなどへの問い合わせに置き換えます。ここで示す3回の反復は動作理解用であり、最適化性能の優劣を判定するベンチマークではありません。

## 11. Short full BO loop

同じ責務を3 iterationだけ繰り返します。Notebook用なのでbenchmarkではなく、状態遷移を確認するための小さなloopです。

In [ ]:
loop_X = train_X.clone()
loop_Y = train_Y.clone()
history = []

for iteration in range(3):
    loop_model = SingleTaskGP(train_X=loop_X, train_Y=loop_Y)
    loop_model.fit()
    loop_model.eval()

    loop_acqf = qLogExpectedImprovement(model=loop_model, best_f=loop_Y.max())
    next_X, next_acq_value = optimize_acqf(
        acq_function=loop_acqf,
        bounds=bounds,
        q=1,
        num_restarts=5,
        raw_samples=64,
    )
    next_Y = objective(next_X)
    history.append((next_X.item(), next_Y.item(), next_acq_value.item()))
    loop_X = torch.cat([loop_X, next_X], dim=0)
    loop_Y = torch.cat([loop_Y, next_Y], dim=0)

for iteration, (x_value, y_value, acquisition_value) in enumerate(history, start=1):
    print(
        f"iteration={iteration}: x={x_value:.4f}, y={y_value:.4f}, "
        f"acq={acquisition_value:.4f}"
    )

print("final best observed value:", loop_Y.max().item())

## 12. BO loop visualization

In [ ]:
grid_Y = objective(test_X).squeeze(-1)

plt.figure(figsize=(8, 4))
plt.plot(test_X.squeeze(-1), grid_Y, label="objective")
plt.scatter(train_X.squeeze(-1), train_Y.squeeze(-1), label="initial observations")
plt.scatter(
    loop_X[train_X.shape[0] :].squeeze(-1),
    loop_Y[train_Y.shape[0] :].squeeze(-1),
    marker="x",
    s=70,
    label="BO evaluations",
)
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## 13. When to use this method

`SingleTaskGP`は、単一タスク・連続変数中心・Exact GPが現実的なデータ量のBOにおける基本モデルです。

- カテゴリ変数を明示的に扱う: `MixedSingleTaskGP`
- 複数タスクの相関を使う: MultiTask / Kronecker系
- fidelityと評価コストを扱う: MultiFidelity系
- 高次元で少数の重要変数を仮定する: SAAS系

## 14. Limitations and caveats

このNotebookはノイズなしの1次元・単一目的・逐次BOです。batch、constraints、mixed variables、multi-objective、pending points、fantasizationは、それぞれ専用workflowで扱います。

## 15. Related robotorchan components

- `robotorchan.models.SingleTaskGP`: surrogate model
- `robotorchan.optim.optimize_acqf`: acquisition optimization dispatch
- BoTorch `qLogExpectedImprovement`: acquisition function

## 16. References

理論・主要文献は冒頭のTheoryリンクおよび各Theory章の参考文献を参照してください。